In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

current_user = spark.sql("SELECT current_user()").collect()[0][0]
default_workspace_path = f"/Workspace/Users/{current_user}/delta_assignment"

dbutils.widgets.text("base_path", default_workspace_path, "Base Workspace Path")
base_path = dbutils.widgets.get("base_path")

bronze_table_path = f"{base_path}/bronze/earthquakes_bronze"
silver_scd1_path = f"{base_path}/silver/earthquakes_scd1"


In [0]:
df_bronze = spark.read.format("delta").load(bronze_table_path)

# Clean & Deduplicate Intra-Batch
window_spec = Window.partitionBy("event_id").orderBy(F.col("event_time").desc())

# Dynamically drop staging/rescued columns if present in Bronze
drop_cols = [c for c in ["_row_num", "_rescued_data", "UNAUTHORIZED_SENSOR_DATA"] if c in df_bronze.columns]

df_stage = (
    df_bronze
    .filter(F.col("_rescued_data").isNull() & F.col("event_id").isNotNull())
    .withColumn("_row_num", F.row_number().over(window_spec))
    .filter(F.col("_row_num") == 1)
    .drop(*drop_cols)
    .withColumn("_updated_at", F.current_timestamp())
)

# Initialize Silver Table if non-existent
if not DeltaTable.isDeltaTable(spark, silver_scd1_path):
    (
        df_stage.limit(0)
        .write.format("delta")
        .mode("overwrite")
        .save(silver_scd1_path)
    )

# Schema Alignment Check
# If Notebook 04 has already renamed 'place' to 'location_name' in Silver,
# align staging column names before executing MERGE
target_cols = spark.read.format("delta").load(silver_scd1_path).columns

if "location_name" in target_cols and "place" in df_stage.columns:
    df_stage = df_stage.withColumnRenamed("place", "location_name")

# Perform SCD Type 1 Upsert via Delta MERGE
target_table = DeltaTable.forPath(spark, silver_scd1_path)

(
    target_table.alias("silver")
    .merge(
        df_stage.alias("stage"),
        "silver.event_id = stage.event_id"
    )
    .whenMatchedUpdateAll(
        condition="stage.event_time >= silver.event_time"
    )
    .whenNotMatchedInsertAll()
    .execute()
)
print("[SCD1 MERGE SUCCESS] Updated Silver SCD Type 1 table.")

In [0]:
current_user = spark.sql("SELECT current_user()").collect()[0][0]
default_workspace_path = f"/Workspace/Users/{current_user}/delta_assignment"

dbutils.widgets.text("base_path", default_workspace_path, "Base Workspace Path")
base_path = dbutils.widgets.get("base_path")

silver_scd1_path = f"{base_path}/silver/earthquakes_scd1"
df_silver = spark.read.format("delta").load(silver_scd1_path)

total_count = df_silver.count()
distinct_keys = df_silver.select("event_id").distinct().count()

print("=== SILVER SCD TYPE 1 VERIFICATION ===")
print(f"Total Silver Records: {total_count}")
print(f"Distinct Primary Keys (event_id): {distinct_keys}")
print(f"Deduplication Status: {'PASSED' if total_count == distinct_keys else 'FAILED'}")

df_silver.select(
    "event_id", "magnitude", "depth_km", "event_time", "_updated_at"
).show(5, truncate=False)